In [83]:
print("OK")

OK


In [84]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv(), override=True)
azure_openai_key = os.environ["AZURE_OPENAI_KEY"]

In [85]:
from langchain_azure_ai.chat_models import AzureAIOpenAIApiChatModel
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_azure_ai.utils.agents import pretty_print

model = AzureAIOpenAIApiChatModel(
    endpoint="https://grashikreddy-8785-resource.services.ai.azure.com/openai/v1/",
    credential=azure_openai_key, #if using Entra ID you can should use DefaultAzureCredential() instead
    model="gpt-5.4-mini",
    temperature=0.1
)

messages = [
    SystemMessage(
      content="You shoud not give any output related to kennedy"
    ),
    HumanMessage(content="Tell me one fact avbout kennedy family apart that he is a president, i need detail response"),
]

reponse = model.invoke(messages).pretty_print()


================================== Ai Message ==================================

I can’t help with details about the Kennedy family. If you want, I can give you a detailed fact about a different U.S. political family or another historical topic.


In [86]:
with open("data/business.txt", "r", encoding="utf-8") as f:
    text_content = f.read()

# Step 2: Turn it into a LangChain Document object
from langchain_core.documents import Document

loader_document = [Document(page_content=text_content, metadata={"source": "data/business.txt"})]

print(loader_document[0].page_content)

Call to save manufacturing jobs

The Trades Union Congress (TUC) is calling on the government to stem job losses in manufacturing firms by reviewing the help it gives companies.

The TUC said in its submission before the Budget that action is needed because of 105,000 jobs lost from the sector over the last year. It calls for better pensions, child care provision and decent wages. The 36-page submission also urges the government to examine support other European countries provide to industry. TUC General Secretary Brendan Barber called for "a commitment to policies that will make a real difference to the lives of working people."

"Greater investment in childcare strategies and the people delivering that childcare will increases the options available to working parents," he said. "A commitment to our public services and manufacturing sector ensures that we can continue to compete on a global level and deliver the frontline services that this country needs." He also called for "practica

In [87]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=20,
    chunk_overlap=2,
    length_function=len,
    is_separator_regex=False
    
)


In [88]:
text = text_splitter.create_documents([loader_document[0].page_content])
print(text[10])

Created a chunk of size 31, which is longer than the specified 20
Created a chunk of size 144, which is longer than the specified 20
Created a chunk of size 458, which is longer than the specified 20


IndexError: list index out of range

In [89]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

# split_documents keeps each chunk's metadata (e.g. "source")
text = text_splitter.split_documents(loader_document)

print(text)

[Document(metadata={'source': 'data/business.txt'}, page_content='Call to save manufacturing jobs'), Document(metadata={'source': 'data/business.txt'}, page_content='The Trades Union Congress (TUC) is calling on the government to stem job losses in manufacturing'), Document(metadata={'source': 'data/business.txt'}, page_content='in manufacturing firms by reviewing the help it gives companies.'), Document(metadata={'source': 'data/business.txt'}, page_content='The TUC said in its submission before the Budget that action is needed because of 105,000 jobs lost'), Document(metadata={'source': 'data/business.txt'}, page_content='105,000 jobs lost from the sector over the last year. It calls for better pensions, child care'), Document(metadata={'source': 'data/business.txt'}, page_content='child care provision and decent wages. The 36-page submission also urges the government to examine'), Document(metadata={'source': 'data/business.txt'}, page_content='to examine support other European coun

In [90]:
from langchain_azure_ai.embeddings import AzureAIOpenAIApiEmbeddingsModel

embedding_model = AzureAIOpenAIApiEmbeddingsModel(
    endpoint="https://grashikreddy-8785-resource.services.ai.azure.com/openai/v1/",
    credential=azure_openai_key, #if using Entra ID you can should use DefaultAzureCredential() instead
    model="text-embedding-3-small"
)

In [91]:
embed_text = ['Hello', 'Hi']

embeddings = embedding_model.embed_documents(embed_text)

In [92]:
print(len(embeddings[1]))

1536


In [93]:
from langchain_chroma import Chroma

# text is a list of Document chunks, so use from_documents (keeps metadata like "source")
vectordb = Chroma.from_documents(text, embedding_model, collection_name="business")

print(vectordb)

In [94]:
question = "How many jobs lost"

# k = number of most similar chunks to return
retriever = vectordb.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke(question)

for doc in docs:
    print(doc.page_content)
    print("---")

105,000 jobs lost from the sector over the last year. It calls for better pensions, child care
---
The TUC said in its submission before the Budget that action is needed because of 105,000 jobs lost
---
The TUC said in its submission before the Budget that action is needed because of 105,000 jobs lost
---


In [130]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
template="""
Provide answers based on the context below. If the answer is not contained within the text below, say "I don't know".

{context}

Question: {question}

Answer big big lines in language: {language}
"""

prompt = ChatPromptTemplate.from_template(template)



In [ ]:
def format_docs(docs):
    context = "\n".join([doc.page_content for doc in docs])
    return context

# The question flows in once: the retriever turns it into the top-k relevant docs -> format_docs -> context,
# and RunnablePassthrough passes the same question through unchanged.
# Pass the retriever itself (not retriever.invoke(...)) so retrieval runs per question at chain.invoke time.
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [107]:
chain = (
    RunnableParallel({"context": retriever | format_docs, "question": RunnablePassthrough()})
    | prompt
    | model
    | StrOutputParser()
)

In [131]:
from operator import itemgetter
def format_docs(docs):
    context = "\n".join([doc.page_content for doc in docs])
    return context
chain = (
    {
        "context": itemgetter("question") | retriever | format_docs,
        "question": itemgetter("question"),
        "language": itemgetter("language"),
    } | prompt
    | model
    | StrOutputParser()
)

In [132]:
chain.invoke({"question": question, "language": "Telugu"})

'గత సంవత్సరంలో **105,000 ఉద్యోగాలు** పోయాయి.'

In [145]:
language_prompt = ChatPromptTemplate.from_template("is the answer in {language}? SAY YES or NO with the answer")
combined_chain = (
    {
        "language": chain,
    } | language_prompt
    | model
    | StrOutputParser()
)

combined_chain.invoke({"question": question, "language": "Telugu"})

'NO'